In [1]:
print('Polska')

Polska


In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('lab1prdom', value=tx) # nazwa topicu
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1) # wysyłanie danych co sekundę

producer.flush()
producer.close()

Overwriting producer.py


In [21]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'lab1prdom',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

# TWÓJ KOD
# Dla każdej wiadomości: sprawdź czy amount > 3000, jeśli tak — wypisz ALERT
# Format: ALERT: TX0042 | 2345.67 PLN | Warszawa | elektronika

for message in consumer:
    tx = message.value
    tx_am = tx['amount']

    if tx_am > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx_am} | {tx['store']} | {tx['category']}")

consumer_filter.flush()
consumer_filter.close()

Overwriting consumer_filter.py


In [8]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

# Napisz konsumenta, który do słownika eventu dodaje pole risk_level: - amount > 3000 → “HIGH” - amount > 1000 → “MEDIUM” - pozostałe → “LOW”
# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
# Dodaj pole risk_level na podstawie amount
# Wypisz wzbogaconą transakcję

consumer = KafkaConsumer(
    'lab1prdom',
    bootstrap_servers='broker:9092',
    group_id = 'enricher',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value
    tx_am = tx['amount']

    if tx_am > 3000:
        tx_risk = 'HIGH'
    elif tx_am < 3000 and tx_am >= 1000:
        tx_risk = 'MEDIUM'
    else:
        tx_risk = 'LOW'

    tx['risk_level'] = tx_risk

    print(tx)

Overwriting consumer_enrich.py


In [7]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'lab1prdom',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

# TWÓJ KOD
# Dla każdej wiadomości:
#   1. Zwiększ store_counts[store]
#   2. Dodaj amount do total_amount[store]
#   3. Co 10 wiadomości wypisz tabelę:
#      Sklep | Liczba | Suma | Średnia

for message in consumer:
    tx = message.value
    tx_st = tx['store']
    tx_am = tx['amount']

    store_counts[tx_st] += 1
    total_amount[tx_st] = total_amount.get(tx_st, 0) + tx_am
    msg_count += 1

    if msg_count % 10 == 0:
        for store in store_counts:
            print(f"{store} | {store_counts[store]} | {total_amount[store]} | {total_amount[store] / store_counts[store]}")

Overwriting consumer_count.py


In [13]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import Counter
import json

# TWÓJ KOD

consumer = KafkaConsumer(
    'lab1prdom',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

kategorie = ['elektronika', 'odzież', 'żywność', 'książki']
cat_counts = Counter()
total_amount = {}

max_cat = {}
for cat in kategorie:
    max_cat[cat] = 0

min_cat = {}
for cat in kategorie:
    min_cat[cat] = 1000000000000000000

msg_count = 0

for message in consumer:
    tx = message.value
    tx_cat = tx['category']
    tx_am = tx['amount']

    cat_counts[tx_cat] += 1
    total_amount[tx_cat] = total_amount.get(tx_cat, 0) + tx_am

    if tx_am > max_cat[tx_cat]:
        max_cat[tx_cat] = tx_am

    if tx_am < min_cat[tx_cat]:
        min_cat[tx_cat] = tx_am
    
    msg_count += 1

    if msg_count % 10 == 0:
        for cat in cat_counts:
            print(f"{cat} | {cat_counts[cat]} | {total_amount[cat]} | MAX: {max_cat[cat]} | MIN: {min_cat[cat]}")

Overwriting consumer_stats.py


In [27]:
%%file consumer_vel1_alert.py 
from kafka import KafkaConsumer
from collections import Counter
from datetime import datetime
import json

consumer = KafkaConsumer(
    'lab1prdom',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

####################################
### OKNA SKOKOWE, NIE PRZESUWNE! ###
####################################

user_counts = Counter()
user_60_time = {}

for message in consumer:
    tx = message.value
    tx_user = tx['user_id']
    tx_time = datetime.fromisoformat(tx['timestamp'])
    tx_60_alert = 0
    tx_3_alert = 0
    
    if tx_user not in user_60_time:
        user_60_time[tx_user] = tx_time

    diff_time = tx_time - user_60_time[tx_user]
    
    if diff_time.total_seconds() <= 60:
        tx_60_alert = 1
        user_counts[tx_user] += 1
    else:
        user_60_time[tx_user] = tx_time
        user_counts[tx_user] = 1

    if user_counts[tx_user] > 3:
        tx_3_alert = 1
        user_counts[tx_user] = 1
        user_60_time[tx_user] = tx_time

    if tx_60_alert == 1 and tx_3_alert == 1:
        print(f"ALERT! Ponad 3 transakcje dla user: {tx_user} w czasie {diff_time}")

Overwriting consumer_vel1_alert.py


In [28]:
%%file consumer_vel2_alert.py
from kafka import KafkaConsumer
from datetime import datetime
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'lab1prdom',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

######################################
### POPRAWNA ODPOWIEDŹ DO PR. DOM. ###
######################################

user_times = defaultdict(list)

for message in consumer:
    tx = message.value
    tx_user = tx['user_id']
    tx_time = datetime.fromisoformat(tx['timestamp'])
    
    user_times[tx_user].append(tx_time)
    
    tech_list = []
    
    for t in user_times[tx_user]:
        diff_time = tx_time - t
        if diff_time.total_seconds() <= 60:
            tech_list.append(t)

    user_times[tx_user] = tech_list

    if len(user_times[tx_user]) > 3:
        print(f"ALERT! Ponad 3 transakcje dla user: {tx_user}")

Overwriting consumer_vel2_alert.py
